In [1]:
"""Step 1: cut snippets where one speaker talks on their own."""
import json
from collections import Counter
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import soundfile as sf

RAW_DIR = Path("../raw_data")
LABEL_DIR = RAW_DIR / "labels"
SAMPLE_RATE = 16_000
LABEL_FPS = 100

## Read JSON

In [2]:
def read_json(path):
    """Read a JSON file."""
    return json.loads(Path(path).read_text())

## Track conversation

In [3]:
def partner_of(track):
    """The other track of the conversation."""
    return "b" if track == "a" else "a"

## Track if speaker is acually speaking

In [4]:
def speech_mask(conversation_id, track, n_frames):
    """Boolean array, one value per 10 ms: is this speaker saying a word?"""
    path = LABEL_DIR / conversation_id / f"speaker_{track}_transcript.json"
    mask = np.zeros(n_frames, dtype=bool)
    for word in read_json(path)["words"]:
        mask[int(word["start"] * LABEL_FPS):
             int(np.ceil(word["end"] * LABEL_FPS))] = True
    return mask

## True values

In [ ]:
def window_share(mask, starts, width):
    """Share of True values in mask[start:start + width] for each start."""
    cumulative = np.concatenate([[0], np.cumsum(mask)])
    return (cumulative[starts + width] - cumulative[starts]) / width

## Measure loudness

In [5]:

def mic_levels(audio_file, meta_file, start_sec, snippet_sec):
    """Share of 'active' 20 ms frames and the signal-to-noise ratio (dB)."""
    noise_floor = read_json(meta_file).get(
        "audio_metrics", {}).get("noise_floor_dbfs", -60)
    audio, _ = sf.read(audio_file, start=int(start_sec * SAMPLE_RATE),
                       frames=int(snippet_sec * SAMPLE_RATE), dtype="float32")
    rms = librosa.feature.rms(y=audio, frame_length=640, hop_length=320)[0]
    level_db = 20 * np.log10(rms + 1e-9)
    active_share = float((level_db > noise_floor + 15).mean())
    snr_db = float(np.percentile(level_db, 90) - noise_floor)
    return active_share, snr_db

## Picking the snippets

  A window is kept only if all four checks pass:
    1. the speaker talks at least `min_speech` of the time (transcript),
    2. the partner says no words (transcript),
    3. the partner's mic is active at most `max_partner_active` (audio),
    4. the speaker is at least `min_snr_db` above their background noise.

In [11]:
def make_snippets(tracks, snippet_sec=4, per_speaker=10, min_speech=0.7,
                  max_partner_active=0.05, min_snr_db=15, skip_start_sec=30,
                  seed=42):

    rng = np.random.default_rng(seed)
    width = int(snippet_sec * LABEL_FPS)
    rejected = Counter()
    rows = []

    for speaker_id, speaker_tracks in tracks.groupby("speaker_id"):
        # 1 + 2: candidate windows from the transcripts, per conversation
        pools = []
        for row in speaker_tracks.sort_values("conversation_id").itertuples():
            conv_dir = RAW_DIR / row.split / row.conversation_id
            own_file = conv_dir / f"speaker_{row.track}.flac"
            n_frames = int(sf.info(own_file).duration * LABEL_FPS)
            own = speech_mask(row.conversation_id, row.track, n_frames)
            partner = speech_mask(row.conversation_id,
                                  partner_of(row.track), n_frames)
            starts = np.arange(skip_start_sec * LABEL_FPS,
                               n_frames - width, LABEL_FPS)
            if len(starts) == 0:
                continue
            ok = ((window_share(own, starts, width) >= min_speech)
                  & (window_share(partner, starts, width) == 0))
            if ok.any():
                shuffled = list(rng.permutation(starts[ok] / LABEL_FPS))
                pools.append((row, shuffled, []))

        # 3 + 4: audio checks, taking turns between conversations
        n_chosen = 0
        while n_chosen < per_speaker and any(pool[1] for pool in pools):
            for row, candidates, picked in pools:
                while candidates and n_chosen < per_speaker:
                    start = float(candidates.pop())
                    if any(abs(start - p) < snippet_sec for p in picked):
                        continue                          # overlaps
                    labels = LABEL_DIR / row.conversation_id
                    conv_dir = RAW_DIR / row.split / row.conversation_id
                    other = partner_of(row.track)
                    partner_active, _ = mic_levels(
                        conv_dir / f"speaker_{other}.flac",
                        labels / f"speaker_{other}_meta.json",
                        start, snippet_sec)
                    if partner_active > max_partner_active:
                        rejected["partner mic"] += 1
                        continue
                    _, own_snr = mic_levels(
                        conv_dir / f"speaker_{row.track}.flac",
                        labels / f"speaker_{row.track}_meta.json",
                        start, snippet_sec)
                    if own_snr < min_snr_db:
                        rejected["own voice too quiet"] += 1
                        continue
                    picked.append(start)
                    n_chosen += 1
                    rows.append({"speaker_id": speaker_id,
                                 "split": row.split,
                                 "conversation_id": row.conversation_id,
                                 "track": row.track,
                                 "start_sec": start,
                                 "end_sec": start + snippet_sec})
                    break                                 # next conversation

    return pd.DataFrame(rows), rejected

## Read the audio of the snippets

In [8]:
def load_snippet(snippet):
    """Read the audio of one snippet (a row of make_snippets' output)."""
    path = (RAW_DIR / snippet.split / snippet.conversation_id
            / f"speaker_{snippet.track}.flac")
    n_frames = int((snippet.end_sec - snippet.start_sec) * SAMPLE_RATE)
    audio, _ = sf.read(path, start=int(snippet.start_sec * SAMPLE_RATE),
                       frames=n_frames, dtype="float32")
    return audio

In [12]:
snippets, rejected = make_snippets(tracks, snippet_sec=4, per_speaker=10)
audio = load_snippet(snippets.iloc[0])          # (64000,) = 4 s at 16 kHz

NameError: name 'tracks' is not defined